### Univariate Analysis

In [ ]:
from scipy import stats
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

# get the figure directory
path = Path('/content/drive/MyDrive/hydromind')
raw_fig_path = path / "figures/raw"
raw_fig_path.mkdir(parents=True, exist_ok=True)
processed_fig_path = path / "figures/processed"
processed_fig_path.mkdir(parents=True, exist_ok=True)

def univariate_analysis(series: pl.Series, name: str, expected_dist: str = None):
    # compute summary stats + histogram + KDE for a numerice series
    arr = series.to_numpy()

    summary = {
        "name": name,
        "mean": np.mean(arr),
        "mode": stats.mode(arr),
        "median": np.median(arr),
        "std": np.std(arr),
        "skewness": stats.skew(arr),
        "kurtosis": stats.kurtosis(arr),
        "pct": np.percentile(arr, [1, 5, 25, 50, 75, 95, 99]),
        "min": np.min(arr),
        "max": np.max(arr),
    }

    fig, axes = plt.subplots(1, 2, figsize=(12,4))

    # histogram + kde
    axes[0].hist(arr, bins=50, density=True, alpha=0.6, color="steelblue")
    kde_x = np.linspace(arr.min(), arr.max(), 300)
    kde = stats.gaussian_kde(arr)
    axes[0].plot(kde_x, kde(kde_x), color="crimson", lw=2)
    axes[0].set_title(f"{name} - Histogram + KDE")

    # QQ plot against expected distribution
    if expected_dist == "gamma":
        stats.probplot(arr[arr > 0], dist=stats.gamma, sparams=(stats.gamma.fit(arr[arr > 0])[0],), plot=axes[1])
    else:
        stats.probplot(arr, plot=axes[1])
    
    axes[1].set_title(f"{name} - QQ Plot")
    plt.tight_layout()
    plt.savefig(f"{processed_fig_path}/{name}_univariate.png", dpi=150)
    plt.close()

    return summary

### Occupancy Count Univariate Analysis

In [ ]:
# load data dicts
%store -r df_north
%store -r df_south

df_north_occupancy_summary = univariate_analysis(series=df_north["household"]["occupancy_count"], name="Occupancy Count (North Hemisphere)", expected_dist="gamma")
df_south_occupancy_summary = univariate_analysis(series=df_south["household"]["occupancy_count"], name="Occupancy Count (South Hemisphere)", expected_dist="gamma")

print(df_north_occupancy_summary)
print(df_south_occupancy_summary)

### Appliance Efficiency Univariate Analysis

In [ ]:
df_north_appliance_summary = univariate_analysis(series=df_north["household"]["appliance_efficiency_score"], name="Appliance Efficiency Score (North Hemisphere)", expected_dist="gamma")
df_south_appliance_summary = univariate_analysis(series=df_south["household"]["appliance_efficiency_score"], name="Appliance Efficiency Score (South Hemisphere)", expected_dist="gamma")

print(df_north_appliance_summary)
print(df_south_appliance_summary)

### Daily Max Temp Univariate Analysis

In [ ]:
df_north_temp_summary = univariate_analysis(series=df_north["environment"]["daily_max_temp_celsius"], name="Daily Max Temperature (North Hemisphere)", expected_dist="gamma")
df_south_temp_summary = univariate_analysis(series=df_south["environment"]["daily_max_temp_celsius"], name="Daily Max Temperature (South Hemisphere)", expected_dist="gamma")

print(df_north_temp_summary)
print(df_south_temp_summary)

### Daily Rainfall Univariate Analysis

In [ ]:
df_north_rainfall_summary = univariate_analysis(series=df_north["environment"]["daily_rainfall_mm"], name="Daily Rainfall (North Hemisphere)", expected_dist="gamma")
df_south_rainfall_summary = univariate_analysis(series=df_south["environment"]["daily_rainfall_mm"], name="Daily Rainfall (South Hemisphere)", expected_dist="gamma")

print(df_north_rainfall_summary)
print(df_south_rainfall_summary)

### Water Usage Univariate Analysis

In [ ]:
def water_usage_univariate(water_usage: pl.DataFrame, name: str, expected_dist: str = None):
    flat_df = water_usage.unpivot(value_name="water_usage_measurement").select("water_usage_measurement")

    # compute global stats
    arr = flat_df["water_usage_measurement"].to_numpy()

    summary = {
        "name": name,
        "null_count": np.isnan(arr).sum(),
        "unique_values": len(np.unique(arr)),
        "mean": np.mean(arr),
        "mode": stats.mode(arr),
        "median": np.median(arr),
        "std": np.std(arr),
        "skewness": stats.skew(arr),
        "kurtosis": stats.kurtosis(arr),
        "pct": np.percentile(arr, [1, 5, 25, 50, 75, 95, 99]),
        "min": np.min(arr),
        "max": np.max(arr),
    }

    fig, axes = plt.subplots(1, 2, figsize=(12,4))

    # histogram + kde
    axes[0].hist(arr, bins=50, density=True, alpha=0.6, color="steelblue")
    plot_sample = np.random.choice(arr, size=100_000, replace=False)
    kde_x = np.linspace(arr.min(), arr.max(), 300)
    kde = stats.gaussian_kde(plot_sample)
    axes[0].plot(kde_x, kde(kde_x), color="crimson", lw=2)
    axes[0].set_title(f"{name} - Histogram + KDE")

    # QQ plot against expected distribution
    if expected_dist == "gamma":
        stats.probplot(plot_sample[plot_sample > 0], dist=stats.gamma, sparams=(stats.gamma.fit(plot_sample[plot_sample > 0])[0],), plot=axes[1])
    else:
        stats.probplot(plot_sample, plot=axes[1])
    
    axes[1].set_title(f"{name} - QQ Plot")
    plt.tight_layout()
    plt.savefig(f"{fig_path}/{name}_univariate.png", dpi=150)
    plt.close()

    return summary

df_north_water_usage_summary = water_usage_univariate(water_usage=df_north["water_usage"], name="Daily Water Usage (North Hemisphere)", expected_dist="gamma")
df_south_water_usage_summary = water_usage_univariate(water_usage=df_south["water_usage"], name="Daily Water Usage (South Hemisphere)", expected_dist="gamma")

print(df_north_water_usage_summary)
print(df_south_water_usage_summary)

### Log Per Capita Usage

In [ ]:
%store -r df_north_processed
%store -r df_south_processed

print(univariate_analysis(series=df_north_processed["log_per_capita_usage"], name="Log Per Capita Usage (North Hemisphere)", expected_dist="gamma"))
print(univariate_analysis(series=df_south_processed["log_per_capita_usage"], name="Log Per Capita Usage (South Hemisphere)", expected_dist="gamma"))

### Dry Day Spike Factor

In [ ]:
print(univariate_analysis(series=df_north_processed["dry_day_spike_factor"], name="Dry Day Spike Factor (North Hemisphere)", expected_dist="gamma"))
print(univariate_analysis(series=df_south_processed["dry_day_spike_factor"], name="Dry Day Spike Factor (South Hemisphere)", expected_dist="gamma"))

### Drought Responsiveness Index

In [ ]:
print(univariate_analysis(series=df_north_processed["drought_responsiveness_index"], name="Drought Responsiveness Index (North Hemisphere)", expected_dist="gamma"))
print(univariate_analysis(series=df_south_processed["drought_responsiveness_index"], name="Drought Responsiveness Index (South Hemisphere)", expected_dist="gamma"))

### Water Usage CV

In [ ]:
print(univariate_analysis(series=df_north_processed["water_usage_cv"], name="Water Usage CV (North Hemisphere)", expected_dist="gamma"))
print(univariate_analysis(series=df_south_processed["water_usage_cv"], name="Water Usage CV (South Hemisphere)", expected_dist="gamma"))

### Efficiency Penalty Ratio

In [ ]:
print(univariate_analysis(series=df_north_processed["efficiency_penalty_ratio"], name="Efficiency Penalty Ratio (North Hemisphere)", expected_dist="gamma"))
print(univariate_analysis(series=df_south_processed["efficiency_penalty_ratio"], name="Efficiency Penalty Ratio (South Hemisphere)", expected_dist="gamma"))

### Temp Sensitivity Corr

In [ ]:
print(univariate_analysis(series=df_north_processed["temp_sensitivity_corr"], name="Temp Sensitivity Corr (North Hemisphere)", expected_dist="gamma"))
print(univariate_analysis(series=df_south_processed["temp_sensitivity_corr"], name="Temp Sensitivity Corr (South Hemisphere)", expected_dist="gamma"))

### Weekend Weekday Ratio

In [ ]:
print(univariate_analysis(series=df_north_processed["weekend_weekday_ratio"], name="Weekend Weekday Ratio (North Hemisphere)", expected_dist="gamma"))
print(univariate_analysis(series=df_south_processed["weekend_weekday_ratio"], name="Weekend Weekday Ratio (South Hemisphere)", expected_dist="gamma"))

### Landscape Demand Index

In [ ]:
print(univariate_analysis(series=df_north_processed["landscape_demand_index"], name="Landscape Demand Index (North Hemisphere)", expected_dist="gamma"))
print(univariate_analysis(series=df_south_processed["landscape_demand_index"], name="Landscape Demand Index (South Hemisphere)", expected_dist="gamma"))

### Seasonal Amplitude Ratio

In [ ]:
print(univariate_analysis(series=df_north_processed["seasonal_amplitude_ratio"], name="Seasonal Amplitude Ratio (North Hemisphere)", expected_dist="gamma"))
print(univariate_analysis(series=df_south_processed["seasonal_amplitude_ratio"], name="Seasonal Amplitude Ratio (South Hemisphere)", expected_dist="gamma"))

### Baseline Peak Ratio

In [ ]:
print(univariate_analysis(series=df_north_processed["baseline_peak_ratio"], name="Baseline Peak Ratio (North Hemisphere)", expected_dist="gamma"))
print(univariate_analysis(series=df_south_processed["baseline_peak_ratio"], name="Baseline Peak Ratio (South Hemisphere)", expected_dist="gamma"))